In [ ]:
#Setup for Target Data:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Load PM2.5 target dataset
pm25_path = "/content/drive/MyDrive/ResearchPG/Datasets/Daily_Data.csv"
pm25 = pd.read_csv(pm25_path)

# Check columns
print("Columns:", pm25.columns.tolist())

# Convert date column
pm25["Date"] = pd.to_datetime(pm25["Date"])

# Select Santa Cruz site
target_site_id = 60870007
target_site = pm25[pm25["Site ID"] == target_site_id].copy()

# Get latitude and longitude from the correct columns
site_lat = target_site["Site Latitude"].iloc[0]
site_lon = target_site["Site Longitude"].iloc[0]

print("Target Site ID:", target_site_id)
print("Latitude:", site_lat)
print("Longitude:", site_lon)

# Keep only needed columns
target_site = target_site[
    ["Date", "Site ID", "Site Latitude", "Site Longitude", "Daily Mean PM2.5 Concentration"]
].copy()

target_site = target_site.rename(columns={
    "Site Latitude": "Latitude",
    "Site Longitude": "Longitude",
    "Daily Mean PM2.5 Concentration": "pm25"
})

print(target_site.head())
print("Total rows:", len(target_site))

Mounted at /content/drive
Columns: ['Date', 'Source', 'Site ID', 'POC', 'Daily Mean PM2.5 Concentration', 'Units', 'Daily AQI Value', 'Local Site Name', 'Daily Obs Count', 'Percent Complete', 'AQS Parameter Code', 'AQS Parameter Description', 'Method Code', 'Method Description', 'CBSA Code', 'CBSA Name', 'State FIPS Code', 'State', 'County FIPS Code', 'County', 'Site Latitude', 'Site Longitude']
Target Site ID: 60870007
Latitude: 36.98332
Longitude: -121.98822
        Date   Site ID  Latitude  Longitude  pm25
0 2025-01-01  60870007  36.98332 -121.98822   7.3
1 2025-01-02  60870007  36.98332 -121.98822   6.2
2 2025-01-03  60870007  36.98332 -121.98822   5.6
3 2025-01-04  60870007  36.98332 -121.98822   5.5
4 2025-01-05  60870007  36.98332 -121.98822   7.2
Total rows: 360


In [ ]:
#Authenticate the Earth Engine API:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='earth-engine-api-492523') #Add your project ID in the quotes.

In [ ]:
#Create a PM2.5 vegetation region around the site and load MODIS NDVI (Google Earth Engine's Vegetation Data)

#Create a point from the PM2.5 site coordinates:
site_point = ee.Geometry.Point([site_lon, site_lat])

#Create a region around the site
#50000 meters = 50 km buffer
region = site_point.buffer(50000).bounds()

#Load MODIS NDVI for 2025 (or whatever your dates are for your dataset:)
modis = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .filterDate("2025-01-01", "2026-01-01")
    .select("NDVI")
)

In [ ]:
#Export the vegetation data as a CSV time series:
def image_to_feature(img):
    stats = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=250,
        maxPixels=1e13
    )
    return ee.Feature(None, {
        "date": ee.Date(img.get("system:time_start")).format("YYYY-MM-dd"),
        "NDVI": stats.get("NDVI")
    })

ndvi_fc = modis.map(image_to_feature)

task = ee.batch.Export.table.toDrive(
    collection=ndvi_fc,
    description="NDVI_timeseries",
    folder="EarthEngineExports",
    fileFormat="CSV"
)

task.start()
print("Export started!")

Export started!


In [ ]:
#Check export status:
print(task.status())

{'state': 'READY', 'description': 'NDVI_timeseries', 'priority': 100, 'creation_timestamp_ms': 1781726846443, 'update_timestamp_ms': 1781726846443, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': '5GIBAWHGFH6ANOWWTVI3KVD3', 'name': 'projects/earth-engine-api-492523/operations/5GIBAWHGFH6ANOWWTVI3KVD3'}


In [ ]:
#Load the exported NDVI CSV from Drive:
import pandas as pd

#Change this path if needed:
ndvi_path = "/content/drive/MyDrive/EarthEngineExports/NDVI_timeseries.csv"

ndvi = pd.read_csv(ndvi_path)
print(ndvi.head())
print(ndvi.columns)
print("Original rows:", len(ndvi))

  system:index         NDVI        date  \
0   2025_01_01  5852.318122  2025-01-01   
1   2025_01_17  5819.383448  2025-01-17   
2   2025_02_02  5947.415988  2025-02-02   
3   2025_02_18  5941.659118  2025-02-18   
4   2025_03_06  6319.327080  2025-03-06   

                                     .geo  
0  {"type":"MultiPoint","coordinates":[]}  
1  {"type":"MultiPoint","coordinates":[]}  
2  {"type":"MultiPoint","coordinates":[]}  
3  {"type":"MultiPoint","coordinates":[]}  
4  {"type":"MultiPoint","coordinates":[]}  
Index(['system:index', 'NDVI', 'date', '.geo'], dtype='object')
Original rows: 23


In [ ]:
print(ndvi.columns)
print(ndvi.head())

Index(['system:index', 'NDVI', 'date', '.geo'], dtype='object')
  system:index         NDVI        date  \
0   2025_01_01  5852.318122  2025-01-01   
1   2025_01_17  5819.383448  2025-01-17   
2   2025_02_02  5947.415988  2025-02-02   
3   2025_02_18  5941.659118  2025-02-18   
4   2025_03_06  6319.327080  2025-03-06   

                                     .geo  
0  {"type":"MultiPoint","coordinates":[]}  
1  {"type":"MultiPoint","coordinates":[]}  
2  {"type":"MultiPoint","coordinates":[]}  
3  {"type":"MultiPoint","coordinates":[]}  
4  {"type":"MultiPoint","coordinates":[]}  


In [ ]:
# Convert all the NDVI data to daily data for 2025

# Convert the date column
ndvi["date"] = pd.to_datetime(ndvi["date"])

# Keep only the columns you actually need
ndvi = ndvi[["date", "NDVI"]].copy()

# Convert NDVI to numeric
ndvi["NDVI"] = pd.to_numeric(ndvi["NDVI"], errors="coerce")

# Set as index
ndvi = ndvi.set_index("date")

# Get the full daily date range for 2025
full_dates = pd.date_range(start="2025-01-01", end="2025-12-31")
ndvi = ndvi.reindex(full_dates)

# Rename the index
ndvi.index.name = "date"

# Insert missing values
ndvi_daily = ndvi.interpolate(method="linear")

# Reset the index
ndvi_daily = ndvi_daily.reset_index()

print(ndvi_daily.head())
print(ndvi_daily.tail())
print("Total rows:", len(ndvi_daily))
print(ndvi_daily.dtypes)

        date         NDVI
0 2025-01-01  5852.318122
1 2025-01-02  5850.259705
2 2025-01-03  5848.201288
3 2025-01-04  5846.142871
4 2025-01-05  5844.084453
          date         NDVI
360 2025-12-27  6264.444437
361 2025-12-28  6264.444437
362 2025-12-29  6264.444437
363 2025-12-30  6264.444437
364 2025-12-31  6264.444437
Total rows: 365
date    datetime64[ns]
NDVI           float64
dtype: object


In [ ]:
#Save the daily vegetation data:
ndvi_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/NDVI_daily.csv"
ndvi_daily.to_csv(ndvi_daily_path, index=False)

print("Saved daily NDVI to:", ndvi_daily_path)

Saved daily NDVI to: /content/drive/MyDrive/ResearchPG/Datasets/NDVI_daily.csv


In [ ]:
#Merge the target PM2.5 data with the daily vegetation data:

#When you download your data, make sure your dates match!
target_site["Date"] = pd.to_datetime(target_site["Date"])
ndvi_daily["date"] = pd.to_datetime(ndvi_daily["date"])

#Merge the two datasets:
merged_pm25_ndvi = target_site.merge(
    ndvi_daily,
    left_on = "Date",
    right_on = "date",
    how = "left"

)

#Drop duplicate date column (this is optional, but for the sake of this project, I will do it:)
merged_pm25_ndvi = merged_pm25_ndvi.drop(columns=["date"])

print(merged_pm25_ndvi.head())
print("Merged rows:", len(merged_pm25_ndvi))

        Date   Site ID  Latitude  Longitude  pm25         NDVI
0 2025-01-01  60870007  36.98332 -121.98822   7.3  5852.318122
1 2025-01-02  60870007  36.98332 -121.98822   6.2  5850.259705
2 2025-01-03  60870007  36.98332 -121.98822   5.6  5848.201288
3 2025-01-04  60870007  36.98332 -121.98822   5.5  5846.142871
4 2025-01-05  60870007  36.98332 -121.98822   7.2  5844.084453
Merged rows: 360


In [ ]:
#Save the merged PM2.5 and the vegetation dataset:

merged_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_merged.csv"
merged_pm25_ndvi.to_csv(merged_path, index=False)

print("Saved merged data to:", merged_path)

Saved merged data to: /content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_merged.csv


In [ ]:
#Pull in hourly data for HRRR, which is our weather variables. The first step is to install Herbie and weather dependencies for HRRR
!pip install -q herbie-data==2024.3.0 pandas==2.2.2 cfgrib xarray netcdf4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.2/62.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.4/424.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.5/307.5 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 62.6 MB/s eta 0:00:00


In [ ]:
#Check the version of pandas (just in case!)

import pandas as pd
print(pd.__version__)

2.2.2


In [ ]:
#Load both Wildfire CSVs

import pandas as pd

fire_j1_path = "/content/drive/MyDrive/ResearchPG/Datasets/WildfireData_J1 VIIRS C2 - fire_archive_J1V-C2_729304.csv"
fire_suomi_path = "/content/drive/MyDrive/ResearchPG/Datasets/Wildfire_Data_SUOMI VIIRS C2 - fire_archive_SV-C2_729305.csv"

fire_j1 = pd.read_csv(fire_j1_path)
fire_suomi = pd.read_csv(fire_suomi_path)

print("J1 columns:", fire_j1.columns.tolist())
print("SUOMI columns:", fire_suomi.columns.tolist())
print("J1 rows:", len(fire_j1))
print("SUOMI rows:", len(fire_suomi))

J1 columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
SUOMI columns: ['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'type']
J1 rows: 28619
SUOMI rows: 27808


In [ ]:
#Combine the two datasets

fire_j1["source_name"] = "J1"
fire_suomi["source_name"] = "SUOMI"

fire_all = pd.concat([fire_j1, fire_suomi], ignore_index=True)

print(fire_all.head())
print("Total wildfire rows:", len(fire_all))

   latitude  longitude  brightness  scan  track    acq_date  acq_time  \
0  40.56269 -123.69317      324.13  0.39   0.36  2025-03-01      1012   
1  40.56345 -123.69770      319.53  0.39   0.36  2025-03-01      1012   
2  39.68690 -121.85736      295.60  0.43   0.38  2025-03-01      1012   
3  39.89914 -123.30003      308.64  0.40   0.37  2025-03-01      1012   
4  39.21783 -122.30177      301.19  0.43   0.38  2025-03-01      1013   

  satellite instrument confidence  version  bright_t31   frp daynight  type  \
0       N20      VIIRS          n        2      281.59  3.24        N     0   
1       N20      VIIRS          n        2      277.94  2.23        N     0   
2       N20      VIIRS          n        2      280.26  0.43        N     0   
3       N20      VIIRS          n        2      280.30  0.89        N     0   
4       N20      VIIRS          n        2      279.93  1.00        N     0   

  source_name  
0          J1  
1          J1  
2          J1  
3          J1  
4     

In [ ]:
#Clean the wildfire date and time columns
fire_all["acq_date"] = pd.to_datetime(fire_all["acq_date"])
fire_all["acq_time"] = fire_all["acq_time"].astype(str).str.zfill(4)

fire_all["datetime"] = pd.to_datetime(
    fire_all["acq_date"].dt.strftime("%Y-%m-%d") + " " +
    fire_all["acq_time"].str[:2] + ":" + fire_all["acq_time"].str[2:],
    errors="coerce"
)

print(fire_all[["acq_date", "acq_time", "datetime"]].head())

    acq_date acq_time            datetime
0 2025-03-01     1012 2025-03-01 10:12:00
1 2025-03-01     1012 2025-03-01 10:12:00
2 2025-03-01     1012 2025-03-01 10:12:00
3 2025-03-01     1012 2025-03-01 10:12:00
4 2025-03-01     1013 2025-03-01 10:13:00


In [ ]:
#Filter all Wildfire data near the site:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

fire_all["distance_km"] = haversine(
    site_lat,
    site_lon,
    fire_all["latitude"],
    fire_all["longitude"]
)

fire_nearby = fire_all[fire_all["distance_km"] <= 100].copy()

print(fire_nearby.head())
print("Nearby wildfire rows:", len(fire_nearby))

    latitude  longitude  brightness  scan  track   acq_date acq_time  \
12  37.75710 -121.65826      296.61  0.47   0.39 2025-03-01     1013   
13  37.45664 -121.93284      302.71  0.46   0.39 2025-03-01     1013   
63  37.50398 -121.08484      301.50  0.38   0.36 2025-03-02     0954   
73  36.41363 -121.38177      332.63  0.40   0.37 2025-03-03     2056   
75  36.44277 -121.35264      330.65  0.40   0.37 2025-03-03     2056   

   satellite instrument confidence  version  bright_t31    frp daynight  type  \
12       N20      VIIRS          n        2      282.57   0.62        N     0   
13       N20      VIIRS          n        2      278.27   0.54        N     2   
63       N20      VIIRS          n        2      281.18   0.49        N     0   
73       N20      VIIRS          n        2      296.18  19.44        D     0   
75       N20      VIIRS          n        2      294.85   5.73        D     0   

   source_name            datetime  distance_km  
12          J1 2025-03-01 10:1

In [ ]:
#Aggregate it to daily features
fire_daily = fire_nearby.groupby("acq_date").agg(
    fire_count=("acq_date", "size"),
    mean_frp=("frp", "mean"),
    max_frp=("frp", "max")
).reset_index()

fire_daily = fire_daily.rename(columns={"acq_date": "Date"})

print(fire_daily.head())
print("Daily wildfire rows:", len(fire_daily))

        Date  fire_count  mean_frp  max_frp
0 2025-03-01           5  0.684000     1.10
1 2025-03-02           2  0.645000     0.80
2 2025-03-03           6  5.673333    19.44
3 2025-03-04           2  4.535000     4.97
4 2025-03-07           6  0.740000     1.57
Daily wildfire rows: 232


In [ ]:
#Force all Wildfire data to 2025
full_dates = pd.date_range(start="2025-01-01", end="2025-12-31")

fire_daily = fire_daily.set_index("Date").reindex(full_dates)
fire_daily.index.name = "Date"

fire_daily["fire_count"] = fire_daily["fire_count"].fillna(0)
fire_daily["mean_frp"] = fire_daily["mean_frp"].fillna(0)
fire_daily["max_frp"] = fire_daily["max_frp"].fillna(0)

fire_daily = fire_daily.reset_index()

print(fire_daily.head())
print(fire_daily.tail())
print("Total wildfire daily rows:", len(fire_daily))

        Date  fire_count  mean_frp  max_frp
0 2025-01-01         0.0       0.0      0.0
1 2025-01-02         0.0       0.0      0.0
2 2025-01-03         0.0       0.0      0.0
3 2025-01-04         0.0       0.0      0.0
4 2025-01-05         0.0       0.0      0.0
          Date  fire_count  mean_frp  max_frp
360 2025-12-27         1.0    4.7000     4.70
361 2025-12-28         2.0    0.4600     0.53
362 2025-12-29         8.0    2.2050     6.68
363 2025-12-30         4.0    0.6875     1.02
364 2025-12-31         0.0    0.0000     0.00
Total wildfire daily rows: 365


In [ ]:
#Save the Wildfire Daily Dataset
fire_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/Wildfire_daily.csv"
fire_daily.to_csv(fire_daily_path, index=False)

print("Saved wildfire daily data to:", fire_daily_path)

Saved wildfire daily data to: /content/drive/MyDrive/ResearchPG/Datasets/Wildfire_daily.csv


In [ ]:
#Merge it into the PM2.5 + NDVI dataset:
merged_pm25_ndvi["Date"] = pd.to_datetime(merged_pm25_ndvi["Date"])
fire_daily["Date"] = pd.to_datetime(fire_daily["Date"])

merged_pm25_ndvi_fire = merged_pm25_ndvi.merge(fire_daily, on="Date", how="left")

print(merged_pm25_ndvi_fire.head())
print("Merged rows:", len(merged_pm25_ndvi_fire))

        Date   Site ID  Latitude  Longitude  pm25         NDVI  fire_count  \
0 2025-01-01  60870007  36.98332 -121.98822   7.3  5852.318122         0.0   
1 2025-01-02  60870007  36.98332 -121.98822   6.2  5850.259705         0.0   
2 2025-01-03  60870007  36.98332 -121.98822   5.6  5848.201288         0.0   
3 2025-01-04  60870007  36.98332 -121.98822   5.5  5846.142871         0.0   
4 2025-01-05  60870007  36.98332 -121.98822   7.2  5844.084453         0.0   

   mean_frp  max_frp  
0       0.0      0.0  
1       0.0      0.0  
2       0.0      0.0  
3       0.0      0.0  
4       0.0      0.0  
Merged rows: 360


In [ ]:
#Save PM2.5 + NDVI + Wildfire merged dataset:
merged_fire_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_merged.csv"
merged_pm25_ndvi_fire.to_csv(merged_fire_path, index=False)

print("Saved merged dataset to:", merged_fire_path)

Saved merged dataset to: /content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_merged.csv


In [ ]:
#HRRR Setup
from herbie import Herbie
import pandas as pd
import numpy as np
import xarray as xr
from tqdm import tqdm
import os

#Make sure all of these dates are in datetime!
merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])

#These will come from your first cell!
print("Using site latitude:", site_lat)
print("Using site longitude", site_lon)
print("Date range:", merged_pm25_ndvi_fire["Date"].min(), "to", merged_pm25_ndvi_fire["Date"].max())

 ╭─▌▌Herbie─────────────────────────────────────────────╮
 │ INFO: Created a default config file.                 │
 │ You may view/edit Herbie's configuration here:       │
 │          /root/.config/herbie/config.toml            │
 ╰──────────────────────────────────────────────────────╯

Using site latitude: 36.98332
Using site longitude -121.98822
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00


In [ ]:
# Function to download HRRR weather variables for one day

def get_hrrr_for_date(date, site_lat, site_lon):
    """
    Pulls HRRR surface weather data for one date near the PM2.5 site.
    Uses the 12z run because it gives a consistent daily snapshot.
    """

    try:
        # HRRR analysis run for that date at 12:00 UTC
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        # Pull common surface weather variables
        ds = h.xarray(
            ":(TMP|DPT):2 m|:(UGRD|VGRD):10 m|:GUST:surface|:PRES:surface"
        )

        # Select nearest grid point to your PM2.5 site
        point = ds.herbie.nearest_points(points=[(site_lon, site_lat)])

        row = {
            "Date": pd.to_datetime(date),
        }

        # Extract variables safely
        for var in point.data_vars:
            value = point[var].values

            if np.size(value) > 0:
                row[var] = float(np.ravel(value)[0])

        return row

    except Exception as e:
        print(f"Could not get HRRR for {date}: {e}")
        return {
            "Date": pd.to_datetime(date),
            "hrrr_error": str(e)
        }

In [ ]:
#HRRR data every 5th day because every day will overload the Colab
#Then, later, we can interpolate this data back to daily data.

merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])

all_dates = sorted(merged_pm25_ndvi_fire["Date"].dropna().unique())

#You can change this number. I am using 5, but you can change it to 3 or 4 for more accuracy or to 7 if Colab crashes.
sample_step = 5

dates = all_dates[::sample_step]

print("Original daily data:", len(all_dates))
print("HRRR sampled data:", len(dates))
print("First sampled date:", dates[0])
print("Last sampled date:", dates[-1])

Original daily data: 360
HRRR sampled data: 72
First sampled date: 2025-01-01 00:00:00
Last sampled date: 2025-12-27 00:00:00


In [ ]:
#Lightweight HRRR pull function
# This avoids using xarray, which was causing memory crashes.
# It only downloads a very small subset of HRRR data (2m temperature).

import os
import gc
import pandas as pd
import numpy as np
from herbie import Herbie

def get_hrrr_minimal(date):
    """
    Downloads a very small HRRR subset for a single date.
    Does not load the full dataset into memory.
    """

    row = {"Date": pd.to_datetime(date)}

    try:
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        # Download only temperature at 2 meters above ground
        grib_file = h.download(searchString=":TMP:2 m above ground:")

        # At this stage, we are only confirming the file downloads successfully
        # We are not extracting values yet to avoid memory issues
        row["temp_2m_K"] = np.nan

        # Immediately delete file to prevent disk/memory buildup
        if os.path.exists(grib_file):
            os.remove(grib_file)

        gc.collect()

    except Exception as e:
        print("Failed:", date, e)
        row["temp_2m_K"] = np.nan

    return row

In [ ]:
# Controlled loop to test HRRR downloads safely
# This runs on only a few dates to confirm stability before scaling up.

from tqdm import tqdm

hrrr_rows = []

# Use only a few dates to prevent crashing during testing
test_dates = dates[:3]

for date in tqdm(test_dates):
    row = get_hrrr_minimal(date)
    hrrr_rows.append(row)

    # Save progress after each iteration
    pd.DataFrame(hrrr_rows).to_csv(
        "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_partial_sampled.csv",
        index=False
    )

    print("Saved:", date)

    gc.collect()

hrrr_sampled = pd.DataFrame(hrrr_rows)

print("Completed test run")
print(hrrr_sampled)

  0%|          | 0/3 [00:00<?, ?it/s]

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-01 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [/root/data/hrrr/20250101]


 33%|███▎      | 1/3 [00:01<00:03,  1.93s/it]

Saved: 2025-01-01 00:00:00
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-06 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [/root/data/hrrr/20250106]


 67%|██████▋   | 2/3 [00:03<00:01,  1.52s/it]

Saved: 2025-01-06 00:00:00
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-11 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [/root/data/hrrr/20250111]


100%|██████████| 3/3 [00:04<00:00,  1.55s/it]

Saved: 2025-01-11 00:00:00
Completed test run
        Date  temp_2m_K
0 2025-01-01        NaN
1 2025-01-06        NaN
2 2025-01-11        NaN


In [ ]:
# Function to extract one site value from a small HRRR GRIB file
# This uses xarray only after downloading one small HRRR variable subset.
# This is much safer than loading multiple HRRR variables at once.

import xarray as xr
import numpy as np
import gc
import os

def extract_point_from_grib(grib_file, site_lat, site_lon):
    """
    Opens a small HRRR GRIB file and extracts the nearest grid value
    to the PM2.5 monitoring site.
    """

    ds = None

    try:
        ds = xr.open_dataset(
            grib_file,
            engine="cfgrib",
            backend_kwargs={"indexpath": ""}
        )

        data_vars = list(ds.data_vars)

        if len(data_vars) == 0:
            return np.nan

        var_name = data_vars[0]

        lats = ds["latitude"].values
        lons = ds["longitude"].values

        target_lon = site_lon

        # HRRR longitude may use 0 to 360 instead of -180 to 180
        if np.nanmax(lons) > 180 and target_lon < 0:
            target_lon = target_lon + 360

        distance = (lats - site_lat) ** 2 + (lons - target_lon) ** 2
        y_index, x_index = np.unravel_index(np.nanargmin(distance), distance.shape)

        value = ds[var_name].values[y_index, x_index]

        return float(value)

    except Exception as e:
        print("Extraction failed:", e)
        return np.nan

    finally:
        if ds is not None:
            ds.close()

        gc.collect()

In [ ]:
#Function to extract multiple HRRR weather features for one date
# Each variable is downloaded separately to reduce memory pressure.

hrrr_variable_map = {
    "temp_2m_K": ":TMP:2 m above ground:",
    "dewpoint_2m_K": ":DPT:2 m above ground:",
    "u_wind_10m": ":UGRD:10 m above ground:",
    "v_wind_10m": ":VGRD:10 m above ground:"
}

def get_hrrr_features_for_date(date, site_lat, site_lon):
    """
    Downloads and extracts HRRR weather variables for one date.
    Saves only the nearest point value for the site.
    """

    row = {"Date": pd.to_datetime(date)}

    try:
        h = Herbie(
            pd.to_datetime(date).strftime("%Y-%m-%d 12:00"),
            model="hrrr",
            product="sfc",
            fxx=0
        )

        for feature_name, search_string in hrrr_variable_map.items():
            grib_file = None

            try:
                grib_file = h.download(searchString=search_string)

                value = extract_point_from_grib(
                    grib_file=grib_file,
                    site_lat=site_lat,
                    site_lon=site_lon
                )

                row[feature_name] = value

            except Exception as e:
                print("Failed variable:", feature_name, "for", date, e)
                row[feature_name] = np.nan

            finally:
                if grib_file is not None and os.path.exists(grib_file):
                    os.remove(grib_file)

                gc.collect()

    except Exception as e:
        print("Failed date:", date, e)
        row["hrrr_error"] = str(e)

    return row

In [ ]:
# Pull sampled HRRR weather values
# This uses the sampled dates from Cell 24.
# Start with sample_step = 15 in Cell 24.
# If this works, you can later try sample_step = 10 or 7.

hrrr_extracted_path = "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_extracted_sampled.csv"

hrrr_rows = []

for date in tqdm(dates):
    row = get_hrrr_features_for_date(date, site_lat, site_lon)
    hrrr_rows.append(row)

    # Save after every date so progress is not lost
    pd.DataFrame(hrrr_rows).to_csv(hrrr_extracted_path, index=False)

    print("Saved HRRR values for:", pd.to_datetime(date).strftime("%Y-%m-%d"))

    gc.collect()

hrrr_sampled = pd.DataFrame(hrrr_rows)

print("Completed sampled HRRR extraction")
print("Rows:", len(hrrr_sampled))
print("Columns:", hrrr_sampled.columns.tolist())

hrrr_sampled.head()

  0%|          | 0/72 [00:00<?, ?it/s]

✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-01 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws


  1%|▏         | 1/72 [00:09<10:39,  9.01s/it]

Saved HRRR values for: 2025-01-01
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Jan-06 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
Saved HRRR values for: 2025-01-06


  3%|▎         | 2/72 [00:13<07:43,  6.63s/it]

...

 99%|█████████▊| 71/72 [03:41<00:02,  2.68s/it]

Saved HRRR values for: 2025-12-22
✅ Found ┊ model=hrrr ┊ product=sfc ┊ 2025-Dec-27 12:00 UTC F00 ┊ GRIB2 @ aws ┊ IDX @ aws
👨🏻‍🏭 Created directory: [/root/data/hrrr/20251227]


100%|██████████| 72/72 [03:44<00:00,  3.12s/it]

Saved HRRR values for: 2025-12-27
Completed sampled HRRR extraction
Rows: 72
Columns: ['Date', 'temp_2m_K', 'dewpoint_2m_K', 'u_wind_10m', 'v_wind_10m']


,Date,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m
0,2025-01-01,278.281555,271.310364,-0.777533,-1.541708
1,2025-01-06,280.569580,277.245850,-1.258052,-1.107056
2,2025-01-11,279.896637,277.852112,-2.612053,-0.625494
3,2025-01-16,278.697449,270.374786,-1.154705,-1.681899
4,2025-01-21,278.448914,268.345551,-0.384708,-2.712133


In [ ]:
# Clean HRRR features and create useful weather variables

hrrr_daily_clean = hrrr_sampled.copy()

hrrr_daily_clean["Date"] = pd.to_datetime(hrrr_daily_clean["Date"])

if "hrrr_error" in hrrr_daily_clean.columns:
    hrrr_daily_clean = hrrr_daily_clean.drop(columns=["hrrr_error"])

# Convert Kelvin to Celsius
if "temp_2m_K" in hrrr_daily_clean.columns:
    hrrr_daily_clean["temp_2m_C"] = hrrr_daily_clean["temp_2m_K"] - 273.15

if "dewpoint_2m_K" in hrrr_daily_clean.columns:
    hrrr_daily_clean["dewpoint_2m_C"] = hrrr_daily_clean["dewpoint_2m_K"] - 273.15

# Calculate wind speed from u and v wind components
if "u_wind_10m" in hrrr_daily_clean.columns and "v_wind_10m" in hrrr_daily_clean.columns:
    hrrr_daily_clean["wind_speed_10m"] = np.sqrt(
        hrrr_daily_clean["u_wind_10m"] ** 2 + hrrr_daily_clean["v_wind_10m"] ** 2
    )

print("Cleaned HRRR columns:")
print(hrrr_daily_clean.columns.tolist())

hrrr_daily_clean.head()

Cleaned HRRR columns:
['Date', 'temp_2m_K', 'dewpoint_2m_K', 'u_wind_10m', 'v_wind_10m', 'temp_2m_C', 'dewpoint_2m_C', 'wind_speed_10m']


,Date,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m,temp_2m_C,dewpoint_2m_C,wind_speed_10m
0,2025-01-01,278.281555,271.310364,-0.777533,-1.541708,5.131555,-1.839636,1.726679
1,2025-01-06,280.569580,277.245850,-1.258052,-1.107056,7.419580,4.095850,1.675788
2,2025-01-11,279.896637,277.852112,-2.612053,-0.625494,6.746637,4.702112,2.685901
3,2025-01-16,278.697449,270.374786,-1.154705,-1.681899,5.547449,-2.775214,2.040129
4,2025-01-21,278.448914,268.345551,-0.384708,-2.712133,5.298914,-4.804449,2.739282


In [ ]:
#Convert sampled HRRR data back into daily data
# Your main PM2.5 dataset is already daily.
# This fills HRRR values for the missing days between sampled dates.

full_date_range = pd.DataFrame({
    "Date": pd.date_range(
        start=merged_pm25_ndvi_fire["Date"].min(),
        end=merged_pm25_ndvi_fire["Date"].max(),
        freq="D"
    )
})

hrrr_daily_filled = full_date_range.merge(
    hrrr_daily_clean,
    on="Date",
    how="left"
)

numeric_cols = hrrr_daily_filled.select_dtypes(include="number").columns

hrrr_daily_filled[numeric_cols] = hrrr_daily_filled[numeric_cols].interpolate(
    method="linear"
)

hrrr_daily_filled[numeric_cols] = hrrr_daily_filled[numeric_cols].bfill().ffill()

print("Daily HRRR rows:", len(hrrr_daily_filled))
print("Missing values after filling:")
print(hrrr_daily_filled.isna().sum())

hrrr_daily_filled.head()

Daily HRRR rows: 365
Missing values after filling:
Date              0
temp_2m_K         0
dewpoint_2m_K     0
u_wind_10m        0
v_wind_10m        0
temp_2m_C         0
dewpoint_2m_C     0
wind_speed_10m    0
dtype: int64


,Date,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m,temp_2m_C,dewpoint_2m_C,wind_speed_10m
0,2025-01-01,278.281555,271.310364,-0.777533,-1.541708,5.131555,-1.839636,1.726679
1,2025-01-02,278.739160,272.497461,-0.873636,-1.454778,5.589160,-0.652539,1.716501
2,2025-01-03,279.196765,273.684558,-0.969740,-1.367847,6.046765,0.534558,1.706323
3,2025-01-04,279.654370,274.871655,-1.065844,-1.280917,6.504370,1.721655,1.696145
4,2025-01-05,280.111975,276.058752,-1.161948,-1.193986,6.961975,2.908752,1.685967


In [ ]:
# Save daily HRRR dataset

hrrr_daily_path = "/content/drive/MyDrive/ResearchPG/Datasets/HRRR_daily_filled.csv"

hrrr_daily_filled.to_csv(hrrr_daily_path, index=False)

print("Saved daily HRRR dataset to:", hrrr_daily_path)

Saved daily HRRR dataset to: /content/drive/MyDrive/ResearchPG/Datasets/HRRR_daily_filled.csv


In [ ]:
#Merge HRRR with PM2.5 + NDVI + wildfire data

merged_pm25_ndvi_fire["Date"] = pd.to_datetime(merged_pm25_ndvi_fire["Date"])
hrrr_daily_filled["Date"] = pd.to_datetime(hrrr_daily_filled["Date"])

final_merged_dataset = merged_pm25_ndvi_fire.merge(
    hrrr_daily_filled,
    on="Date",
    how="left"
)

print("Final dataset shape:", final_merged_dataset.shape)
print("Final dataset columns:")
print(final_merged_dataset.columns.tolist())

final_merged_dataset.head()

Final dataset shape: (360, 16)
Final dataset columns:
['Date', 'Site ID', 'Latitude', 'Longitude', 'pm25', 'NDVI', 'fire_count', 'mean_frp', 'max_frp', 'temp_2m_K', 'dewpoint_2m_K', 'u_wind_10m', 'v_wind_10m', 'temp_2m_C', 'dewpoint_2m_C', 'wind_speed_10m']


,Date,Site ID,Latitude,Longitude,pm25,NDVI,fire_count,mean_frp,max_frp,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m,temp_2m_C,dewpoint_2m_C,wind_speed_10m
0,2025-01-01,60870007,36.98332,-121.98822,7.3,5852.318122,0.0,0.0,0.0,278.281555,271.310364,-0.777533,-1.541708,5.131555,-1.839636,1.726679
1,2025-01-02,60870007,36.98332,-121.98822,6.2,5850.259705,0.0,0.0,0.0,278.739160,272.497461,-0.873636,-1.454778,5.589160,-0.652539,1.716501
2,2025-01-03,60870007,36.98332,-121.98822,5.6,5848.201288,0.0,0.0,0.0,279.196765,273.684558,-0.969740,-1.367847,6.046765,0.534558,1.706323
3,2025-01-04,60870007,36.98332,-121.98822,5.5,5846.142871,0.0,0.0,0.0,279.654370,274.871655,-1.065844,-1.280917,6.504370,1.721655,1.696145
4,2025-01-05,60870007,36.98332,-121.98822,7.2,5844.084453,0.0,0.0,0.0,280.111975,276.058752,-1.161948,-1.193986,6.961975,2.908752,1.685967


In [ ]:
# Save final PM2.5 + NDVI + wildfire + HRRR dataset

final_merged_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_HRRR_merged.csv"

final_merged_dataset.to_csv(final_merged_path, index=False)

print("Saved final merged dataset to:", final_merged_path)

Saved final merged dataset to: /content/drive/MyDrive/ResearchPG/Datasets/PM25_NDVI_Wildfire_HRRR_merged.csv


In [ ]:
#Prepare final dataset for machine learning

import pandas as pd
import numpy as np

model_df = final_merged_dataset.copy()

# Make sure Date is datetime
model_df["Date"] = pd.to_datetime(model_df["Date"])

# Sort by date because this is time-based data
model_df = model_df.sort_values("Date").reset_index(drop=True)

print("Dataset shape:", model_df.shape)
print("Columns:")
print(model_df.columns.tolist())

model_df.head()

Dataset shape: (360, 16)
Columns:
['Date', 'Site ID', 'Latitude', 'Longitude', 'pm25', 'NDVI', 'fire_count', 'mean_frp', 'max_frp', 'temp_2m_K', 'dewpoint_2m_K', 'u_wind_10m', 'v_wind_10m', 'temp_2m_C', 'dewpoint_2m_C', 'wind_speed_10m']


,Date,Site ID,Latitude,Longitude,pm25,NDVI,fire_count,mean_frp,max_frp,temp_2m_K,dewpoint_2m_K,u_wind_10m,v_wind_10m,temp_2m_C,dewpoint_2m_C,wind_speed_10m
0,2025-01-01,60870007,36.98332,-121.98822,7.3,5852.318122,0.0,0.0,0.0,278.281555,271.310364,-0.777533,-1.541708,5.131555,-1.839636,1.726679
1,2025-01-02,60870007,36.98332,-121.98822,6.2,5850.259705,0.0,0.0,0.0,278.739160,272.497461,-0.873636,-1.454778,5.589160,-0.652539,1.716501
2,2025-01-03,60870007,36.98332,-121.98822,5.6,5848.201288,0.0,0.0,0.0,279.196765,273.684558,-0.969740,-1.367847,6.046765,0.534558,1.706323
3,2025-01-04,60870007,36.98332,-121.98822,5.5,5846.142871,0.0,0.0,0.0,279.654370,274.871655,-1.065844,-1.280917,6.504370,1.721655,1.696145
4,2025-01-05,60870007,36.98332,-121.98822,7.2,5844.084453,0.0,0.0,0.0,280.111975,276.058752,-1.161948,-1.193986,6.961975,2.908752,1.685967


In [ ]:
#Identify the PM2.5 target column

possible_targets = [
    "Daily Mean PM2.5 Concentration",
    "PM2.5",
    "pm25",
    "PM25",
    "pm2.5"
]

target_col = None

for col in possible_targets:
    if col in model_df.columns:
        target_col = col
        break

if target_col is None:
    print("Could not automatically find PM2.5 column.")
    print("Available columns:")
    print(model_df.columns.tolist())
else:
    print("Target column:", target_col)

Target column: pm25


In [ ]:
# Create date-based features
# These help the model learn seasonal patterns.

model_df["month"] = model_df["Date"].dt.month
model_df["day_of_year"] = model_df["Date"].dt.dayofyear
model_df["day_of_week"] = model_df["Date"].dt.dayofweek

model_df[["Date", "month", "day_of_year", "day_of_week"]].head()

,Date,month,day_of_year,day_of_week
0,2025-01-01,1,1,2
1,2025-01-02,1,2,3
2,2025-01-03,1,3,4
3,2025-01-04,1,4,5
4,2025-01-05,1,5,6


In [ ]:
#Create lag features
# These use previous PM2.5 values to predict future PM2.5.

model_df["pm25_lag_1"] = model_df[target_col].shift(1)
model_df["pm25_lag_3"] = model_df[target_col].shift(3)
model_df["pm25_lag_7"] = model_df[target_col].shift(7)

model_df["pm25_rolling_3"] = model_df[target_col].rolling(window=3).mean()
model_df["pm25_rolling_7"] = model_df[target_col].rolling(window=7).mean()

model_df.head(10)

,Date,Site ID,Latitude,Longitude,pm25,NDVI,fire_count,mean_frp,max_frp,temp_2m_K,...,dewpoint_2m_C,wind_speed_10m,month,day_of_year,day_of_week,pm25_lag_1,pm25_lag_3,pm25_lag_7,pm25_rolling_3,pm25_rolling_7
0,2025-01-01,60870007,36.98332,-121.98822,7.3,5852.318122,0.0,0.0,0.0,278.281555,...,-1.839636,1.726679,1,1,2,NaN,NaN,NaN,NaN,NaN
1,2025-01-02,60870007,36.98332,-121.98822,6.2,5850.259705,0.0,0.0,0.0,278.739160,...,-0.652539,1.716501,1,2,3,7.3,NaN,NaN,NaN,NaN
2,2025-01-03,60870007,36.98332,-121.98822,5.6,5848.201288,0.0,0.0,0.0,279.196765,...,0.534558,1.706323,1,3,4,6.2,NaN,NaN,6.366667,NaN
3,2025-01-04,60870007,36.98332,-121.98822,5.5,5846.142871,0.0,0.0,0.0,279.654370,...,1.721655,1.696145,1,4,5,5.6,7.3,NaN,5.766667,NaN
4,2025-01-05,60870007,36.98332,-121.98822,7.2,5844.084453,0.0,0.0,0.0,280.111975,...,2.908752,1.685967,1,5,6,5.5,6.2,NaN,6.100000,NaN
5,2025-01-06,60870007,36.98332,-121.98822,6.2,5842.026036,0.0,0.0,0.0,280.569580,...,4.095850,1.675788,1,6,0,7.2,5.6,NaN,6.300000,NaN
6,2025-01-07,60870007,36.98332,-121.98822,1.7,5839.967619,0.0,0.0,0.0,280.434991,...,4.217102,1.877811,1,7,1,6.2,5.5,NaN,5.033333,5.671429
7,2025-01-08,60870007,36.98332,-121.98822,4.8,5837.909202,0.0,0.0,0.0,280.300403,...,4.338354,2.079833,1,8,2,1.7,7.2,7.3,4.233333,5.314286
8,2025-01-09,60870007,36.98332,-121.98822,5.6,5835.850785,0.0,0.0,0.0,280.165814,...,4.459607,2.281856,1,9,3,4.8,6.2,6.2,4.033333,5.228571
9,2025-01-10,60870007,36.98332,-121.98822,5.2,5833.792368,0.0,0.0,0.0,280.031226,...,4.580859,2.483878,1,10,4,5.6,1.7,5.6,5.200000,5.171429


In [ ]:
#Select numeric features for modeling

exclude_cols = ["Date", target_col]

numeric_cols = model_df.select_dtypes(include=["number"]).columns.tolist()

feature_cols = [col for col in numeric_cols if col not in exclude_cols]

print("Number of features:", len(feature_cols))
print("Features used:")
print(feature_cols)

Number of features: 22
Features used:
['Site ID', 'Latitude', 'Longitude', 'NDVI', 'fire_count', 'mean_frp', 'max_frp', 'temp_2m_K', 'dewpoint_2m_K', 'u_wind_10m', 'v_wind_10m', 'temp_2m_C', 'dewpoint_2m_C', 'wind_speed_10m', 'month', 'day_of_year', 'day_of_week', 'pm25_lag_1', 'pm25_lag_3', 'pm25_lag_7', 'pm25_rolling_3', 'pm25_rolling_7']


In [ ]:
# Clean dataset for modeling

ml_df = model_df[["Date", target_col] + feature_cols].copy()

# Remove rows with missing target
ml_df = ml_df.dropna(subset=[target_col])

# Fill missing feature values
ml_df[feature_cols] = ml_df[feature_cols].interpolate(method="linear")
ml_df[feature_cols] = ml_df[feature_cols].bfill().ffill()

# Drop remaining missing rows caused by lag features
ml_df = ml_df.dropna().reset_index(drop=True)

print("ML dataset shape:", ml_df.shape)
print("Missing values:")
print(ml_df.isna().sum())

ml_df.head()

ML dataset shape: (360, 24)
Missing values:
Date              0
pm25              0
Site ID           0
Latitude          0
Longitude         0
NDVI              0
fire_count        0
mean_frp          0
max_frp           0
temp_2m_K         0
dewpoint_2m_K     0
u_wind_10m        0
v_wind_10m        0
temp_2m_C         0
dewpoint_2m_C     0
wind_speed_10m    0
month             0
day_of_year       0
day_of_week       0
pm25_lag_1        0
pm25_lag_3        0
pm25_lag_7        0
pm25_rolling_3    0
pm25_rolling_7    0
dtype: int64


,Date,pm25,Site ID,Latitude,Longitude,NDVI,fire_count,mean_frp,max_frp,temp_2m_K,...,dewpoint_2m_C,wind_speed_10m,month,day_of_year,day_of_week,pm25_lag_1,pm25_lag_3,pm25_lag_7,pm25_rolling_3,pm25_rolling_7
0,2025-01-01,7.3,60870007,36.98332,-121.98822,5852.318122,0.0,0.0,0.0,278.281555,...,-1.839636,1.726679,1,1,2,7.3,7.3,7.3,6.366667,5.671429
1,2025-01-02,6.2,60870007,36.98332,-121.98822,5850.259705,0.0,0.0,0.0,278.739160,...,-0.652539,1.716501,1,2,3,7.3,7.3,7.3,6.366667,5.671429
2,2025-01-03,5.6,60870007,36.98332,-121.98822,5848.201288,0.0,0.0,0.0,279.196765,...,0.534558,1.706323,1,3,4,6.2,7.3,7.3,6.366667,5.671429
3,2025-01-04,5.5,60870007,36.98332,-121.98822,5846.142871,0.0,0.0,0.0,279.654370,...,1.721655,1.696145,1,4,5,5.6,7.3,7.3,5.766667,5.671429
4,2025-01-05,7.2,60870007,36.98332,-121.98822,5844.084453,0.0,0.0,0.0,280.111975,...,2.908752,1.685967,1,5,6,5.5,6.2,7.3,6.100000,5.671429


In [ ]:
# Time-based train/test split
# We use the earlier 80% of dates for training and the later 20% for testing.

split_index = int(len(ml_df) * 0.8)

train_df = ml_df.iloc[:split_index]
test_df = ml_df.iloc[split_index:]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))
print("Train date range:", train_df["Date"].min(), "to", train_df["Date"].max())
print("Test date range:", test_df["Date"].min(), "to", test_df["Date"].max())

Training rows: 288
Testing rows: 72
Train date range: 2025-01-01 00:00:00 to 2025-10-19 00:00:00
Test date range: 2025-10-20 00:00:00 to 2025-12-31 00:00:00


In [ ]:
# Scaled model setup
# This cell rescales the model variables before running any models.
# Features are scaled with MinMaxScaler.
# The target is also scaled for model training, then predictions are converted back to PM2.5 units for evaluation.

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

# Scale X features between 0 and 1
x_scaler = MinMaxScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

# Scale y target between 0 and 1
y_scaler = MinMaxScaler()
y_train_scaled = y_scaler.fit_transform(np.array(y_train).reshape(-1, 1)).ravel()
y_test_scaled = y_scaler.transform(np.array(y_test).reshape(-1, 1)).ravel()

print("X_train_scaled min:", X_train_scaled.min())
print("X_train_scaled max:", X_train_scaled.max())
print("y_train_scaled min:", y_train_scaled.min())
print("y_train_scaled max:", y_train_scaled.max())

X_train_scaled min: 0.0
X_train_scaled max: 1.0000000000000002
y_train_scaled min: 0.0
y_train_scaled max: 1.0


In [ ]:
# Scaled baseline model
# Baseline uses the previous day's PM2.5 value as the prediction.
# The lag values are scaled first, then converted back to PM2.5 units for evaluation.

baseline_lag_raw = np.array(test_df["pm25_lag_1"]).reshape(-1, 1)
baseline_lag_scaled = y_scaler.transform(baseline_lag_raw).ravel()

baseline_preds = y_scaler.inverse_transform(
    baseline_lag_scaled.reshape(-1, 1)
).ravel()

baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_preds))
baseline_r2 = r2_score(y_test, baseline_preds)
print("Scaled Baseline Model Results")
print("MAE:", baseline_mae)
print("RMSE:", baseline_rmse)
print("R²:", baseline_r2)

Scaled Baseline Model Results
MAE: 1.2319444444444443
RMSE: 1.4890806261284548
R²: 0.3703325326548108


In [ ]:
# Scaled Linear Regression model

from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train_scaled)

linear_preds_scaled = linear_model.predict(X_test_scaled)

linear_preds = y_scaler.inverse_transform(
    linear_preds_scaled.reshape(-1, 1)
).ravel()

linear_mae = mean_absolute_error(y_test, linear_preds)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_preds))
linear_r2 = r2_score(y_test, linear_preds)

print("Scaled Linear Regression Results")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R²:", linear_r2)

Scaled Linear Regression Results
MAE: 0.6734097838917541
RMSE: 0.8504494347750086
R²: 0.7946135337909305


In [ ]:
# Scaled Random Forest model

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=8,
    random_state=42
)

rf_model.fit(X_train_scaled, y_train_scaled)

rf_preds_scaled = rf_model.predict(X_test_scaled)

rf_preds = y_scaler.inverse_transform(
    rf_preds_scaled.reshape(-1, 1)
).ravel()

rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print("Scaled Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R²:", rf_r2)

Scaled Random Forest Results
MAE: 0.8306309917925052
RMSE: 1.0315013574080818
R²: 0.6978556784211113


In [ ]:
# Scaled Gradient Boosting model

from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

gb_model.fit(X_train_scaled, y_train_scaled)

gb_preds_scaled = gb_model.predict(X_test_scaled)

gb_preds = y_scaler.inverse_transform(
    gb_preds_scaled.reshape(-1, 1)
).ravel()

gb_mae = mean_absolute_error(y_test, gb_preds)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_preds))
gb_r2 = r2_score(y_test, gb_preds)

print("Scaled Gradient Boosting Results")
print("MAE:", gb_mae)
print("RMSE:", gb_rmse)
print("R²:", gb_r2)

Scaled Gradient Boosting Results
MAE: 0.9621478157116938
RMSE: 1.150420914410025
R²: 0.6241726636225874


In [ ]:
# Scaled Polynomial Ridge Regression model
# Polynomial features create interaction and squared terms.
# Ridge regularization prevents unstable coefficients.

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("ridge", Ridge(alpha=10.0))
])

poly_model.fit(X_train_scaled, y_train_scaled)

poly_preds_scaled = poly_model.predict(X_test_scaled)

poly_preds = y_scaler.inverse_transform(
    poly_preds_scaled.reshape(-1, 1)
).ravel()

poly_mae = mean_absolute_error(y_test, poly_preds)
poly_rmse = np.sqrt(mean_squared_error(y_test, poly_preds))
poly_r2 = r2_score(y_test, poly_preds)

print("Scaled Polynomial Ridge Regression Results")
print("MAE:", poly_mae)
print("RMSE:", poly_rmse)
print("R²:", poly_r2)

Scaled Polynomial Ridge Regression Results
MAE: 0.832871677800794
RMSE: 1.0282672214221387
R²: 0.6997473752446353


In [ ]:
# Scaled MLP model

from sklearn.neural_network import MLPRegressor

mlp_model = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32, 16),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    max_iter=1000,
    random_state=42
)

mlp_model.fit(X_train_scaled, y_train_scaled)

mlp_preds_scaled = mlp_model.predict(X_test_scaled)

mlp_preds = y_scaler.inverse_transform(
    mlp_preds_scaled.reshape(-1, 1)
).ravel()

mlp_mae = mean_absolute_error(y_test, mlp_preds)
mlp_rmse = np.sqrt(mean_squared_error(y_test, mlp_preds))
mlp_r2 = r2_score(y_test, mlp_preds)

print("Scaled MLP Results")
print("MAE:", mlp_mae)
print("RMSE:", mlp_rmse)
print("R²:", mlp_r2)

Scaled MLP Results
MAE: 1.0232084156083916
RMSE: 1.3228932054269582
R²: 0.5030366052423215


In [ ]:
# Scaled MLP feature extraction + Linear Regression
# This uses the hidden-layer representation from the scaled MLP as extra features for Linear Regression.

def get_mlp_features(model, X):
    layer_output = X

    # Pass through all hidden layers, excluding the final output layer
    for i in range(len(model.coefs_) - 1):
        layer_output = np.dot(layer_output, model.coefs_[i]) + model.intercepts_[i]
        layer_output = np.maximum(layer_output, 0)  # ReLU

    return layer_output

mlp_train_features = get_mlp_features(mlp_model, X_train_scaled)
mlp_test_features = get_mlp_features(mlp_model, X_test_scaled)

X_train_mlp_lr = np.hstack([X_train_scaled, mlp_train_features])
X_test_mlp_lr = np.hstack([X_test_scaled, mlp_test_features])

mlp_lr_model = LinearRegression()
mlp_lr_model.fit(X_train_mlp_lr, y_train_scaled)

mlp_lr_preds_scaled = mlp_lr_model.predict(X_test_mlp_lr)

mlp_lr_preds = y_scaler.inverse_transform(
    mlp_lr_preds_scaled.reshape(-1, 1)
).ravel()

mlp_lr_mae = mean_absolute_error(y_test, mlp_lr_preds)
mlp_lr_rmse = np.sqrt(mean_squared_error(y_test, mlp_lr_preds))
mlp_lr_r2 = r2_score(y_test, mlp_lr_preds)

print("Scaled MLP + Linear Regression Results")
print("MAE:", mlp_lr_mae)
print("RMSE:", mlp_lr_rmse)
print("R²:", mlp_lr_r2)

Scaled MLP + Linear Regression Results
MAE: 1.2175551259350639
RMSE: 1.5460364700968414
R²: 0.3212430368384338


In [ ]:
# Scaled ARIMA model
# ARIMA uses only the scaled target series.
# Predictions are converted back to PM2.5 units before evaluation.

from statsmodels.tsa.arima.model import ARIMA

arima_model = ARIMA(
    y_train_scaled,
    order=(3, 1, 2)
)

arima_fit = arima_model.fit()

arima_preds_scaled = arima_fit.forecast(steps=len(y_test_scaled))

arima_preds = y_scaler.inverse_transform(
    np.array(arima_preds_scaled).reshape(-1, 1)
).ravel()

arima_mae = mean_absolute_error(y_test, arima_preds)
arima_rmse = np.sqrt(mean_squared_error(y_test, arima_preds))
arima_r2 = r2_score(y_test, arima_preds)

print("Scaled ARIMA Results")
print("MAE:", arima_mae)
print("RMSE:", arima_rmse)
print("R²:", arima_r2)

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


Scaled ARIMA Results
MAE: 1.5902452335424528
RMSE: 1.8890298069727791
R²: -0.013333616868385967


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
# Scaled SARIMAX model
# SARIMAX uses the scaled target series and scaled external variables.

from statsmodels.tsa.statespace.sarimax import SARIMAX

sarimax_model = SARIMAX(
    y_train_scaled,
    exog=X_train_scaled,
    order=(1, 1, 1),
    seasonal_order=(0, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)

sarimax_fit = sarimax_model.fit(maxiter=200, disp=False)

sarimax_preds_scaled = sarimax_fit.predict(
    start=len(y_train_scaled),
    end=len(y_train_scaled) + len(y_test_scaled) - 1,
    exog=X_test_scaled
)

sarimax_preds = y_scaler.inverse_transform(
    np.array(sarimax_preds_scaled).reshape(-1, 1)
).ravel()

sarimax_mae = mean_absolute_error(y_test, sarimax_preds)
sarimax_rmse = np.sqrt(mean_squared_error(y_test, sarimax_preds))
sarimax_r2 = r2_score(y_test, sarimax_preds)

print("Scaled SARIMAX Results")
print("MAE:", sarimax_mae)
print("RMSE:", sarimax_rmse)
print("R²:", sarimax_r2)

Scaled SARIMAX Results
MAE: 6.237074800790662
RMSE: 6.972388726416878
R²: -12.805050924739469


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [ ]:
# Final comparison table for all scaled models

scaled_results_df = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting",
        "Polynomial Ridge Regression",
        "MLP",
        "MLP + Linear Regression",
        "ARIMA",
        "SARIMAX"
    ],
    "MAE": [
        baseline_mae,
        linear_mae,
        rf_mae,
        gb_mae,
        poly_mae,
        mlp_mae,
        mlp_lr_mae,
        arima_mae,
        sarimax_mae
    ],
    "RMSE": [
        baseline_rmse,
        linear_rmse,
        rf_rmse,
        gb_rmse,
        poly_rmse,
        mlp_rmse,
        mlp_lr_rmse,
        arima_rmse,
        sarimax_rmse
    ],
    "R²": [
        baseline_r2,
        linear_r2,
        rf_r2,
        gb_r2,
        poly_r2,
        mlp_r2,
        mlp_lr_r2,
        arima_r2,
        sarimax_r2
    ]
})

scaled_results_df = scaled_results_df.sort_values("R²", ascending=False)

scaled_results_df

,Model,MAE,RMSE,R²
1,Linear Regression,0.673410,0.850449,0.794614
4,Polynomial Ridge Regression,0.832872,1.028267,0.699747
2,Random Forest,0.830631,1.031501,0.697856
3,Gradient Boosting,0.962148,1.150421,0.624173
5,MLP,1.023208,1.322893,0.503037
0,Baseline,1.231944,1.489081,0.370333
6,MLP + Linear Regression,1.217555,1.546036,0.321243
7,ARIMA,1.590245,1.889030,-0.013334
8,SARIMAX,6.237075,6.972389,-12.805051


In [ ]:
# Save scaled model predictions

scaled_predictions_df = test_df[["Date", target_col]].copy()

scaled_predictions_df["Baseline_Prediction"] = baseline_preds
scaled_predictions_df["Linear_Regression_Prediction"] = linear_preds
scaled_predictions_df["Random_Forest_Prediction"] = rf_preds
scaled_predictions_df["Gradient_Boosting_Prediction"] = gb_preds
scaled_predictions_df["Polynomial_Ridge_Prediction"] = poly_preds
scaled_predictions_df["MLP_Prediction"] = mlp_preds
scaled_predictions_df["MLP_Linear_Regression_Prediction"] = mlp_lr_preds
scaled_predictions_df["ARIMA_Prediction"] = arima_preds
scaled_predictions_df["SARIMAX_Prediction"] = sarimax_preds

scaled_predictions_path = "/content/drive/MyDrive/ResearchPG/Datasets/PM25_scaled_model_predictions.csv"

scaled_predictions_df.to_csv(scaled_predictions_path, index=False)

print("Saved scaled model predictions to:", scaled_predictions_path)

scaled_predictions_df.head()

Saved scaled model predictions to: /content/drive/MyDrive/ResearchPG/Datasets/PM25_scaled_model_predictions.csv


,Date,pm25,Baseline_Prediction,Linear_Regression_Prediction,Random_Forest_Prediction,Gradient_Boosting_Prediction,Polynomial_Ridge_Prediction,MLP_Prediction,MLP_Linear_Regression_Prediction,ARIMA_Prediction,SARIMAX_Prediction
288,2025-10-20,8.5,7.5,7.463308,7.185991,7.142355,6.937699,6.885364,7.375561,6.752695,9.513297
289,2025-10-21,7.1,8.5,7.623811,7.142331,7.102408,7.714864,7.465743,7.632954,6.057713,10.088401
290,2025-10-22,7.0,7.1,7.663670,7.270513,7.034205,7.293552,6.961514,7.682625,5.831406,9.376659
291,2025-10-24,6.1,7.0,6.183960,6.824257,6.962313,6.653590,6.518912,6.353320,5.657005,8.315891
292,2025-10-25,4.2,6.1,5.268978,5.014178,5.174933,5.587253,5.596823,5.123776,5.676815,7.021908


In [ ]:
# Polynomial Regression with Ridge and Lasso penalties

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

results = []

# Test polynomial degrees
for degree in range(1, 5):


    poly_ridge_model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("regressor", Ridge(alpha=1.0))
    ])

    poly_ridge_model.fit(X_train_scaled, y_train)

    ridge_preds = poly_ridge_model.predict(X_test_scaled)
    ridge_mae = mean_absolute_error(y_test, ridge_preds)
    ridge_mse = mean_squared_error(y_test, ridge_preds)
    ridge_rmse = np.sqrt(ridge_mse)
    ridge_r2 = r2_score(y_test, ridge_preds)

    results.append({
        "Penalty": "Ridge",
        "Degree": degree,
        "MAE": ridge_mae,
        "MSE": ridge_mse,
        "RMSE": ridge_rmse,
        "R²": ridge_r2
    })


    poly_lasso_model = Pipeline([
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("regressor", Lasso(alpha=0.001, max_iter=10000))
    ])

    poly_lasso_model.fit(X_train_scaled, y_train)

    lasso_preds = poly_lasso_model.predict(X_test_scaled)
    lasso_mae = mean_absolute_error(y_test, lasso_preds)
    lasso_mse = mean_squared_error(y_test, lasso_preds)
    lasso_rmse = np.sqrt(lasso_mse)
    lasso_r2 = r2_score(y_test, lasso_preds)

    results.append({
        "Penalty": "Lasso",
        "MAE": lasso_mae,
        "Degree": degree,
        "MSE": lasso_mse,
        "RMSE": lasso_rmse,
        "R²": lasso_r2
    })

# Create results table
poly_penalty_results_df = pd.DataFrame(results)

# Sort by RMSE
poly_penalty_results_df = poly_penalty_results_df.sort_values("RMSE")

print(poly_penalty_results_df)

  Penalty  Degree       MAE       MSE      RMSE        R²
1   Lasso       1  0.681950  0.727517  0.852946  0.793406
2   Ridge       2  0.776138  0.919240  0.958770  0.738962
0   Ridge       1  0.791423  0.990235  0.995106  0.718801
3   Lasso       2  0.966914  1.391779  1.179737  0.604774
4   Ridge       3  1.043508  1.711606  1.308284  0.513953
5   Lasso       3  1.118471  1.921883  1.386320  0.454240
7   Lasso       4  1.546112  3.531585  1.879251 -0.002870
6   Ridge       4  1.625412  4.197962  2.048893 -0.192102


In [ ]:
# Compare best polynomial penalty model to all other models

import pandas as pd

best_poly_penalty = poly_penalty_results_df.sort_values("RMSE").iloc[0]

best_poly_model_name = (
    "Polynomial Regression + "
    + best_poly_penalty["Penalty"]
    + " Penalty (Degree "
    + str(int(best_poly_penalty["Degree"]))
    + ")"
)

final_comparison_df = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression",
        best_poly_model_name,
        "Random Forest",
        "Gradient Boosting",
        "MLP",
        "MLP + Linear Regression",
        "ARIMA",
        "SARIMAX"
    ],

    "MAE": [
        baseline_mae,
        linear_mae,
        np.nan,
        rf_mae,
        gb_mae,
        mlp_mae,
        mlp_lr_mae,
        arima_mae,
        sarimax_mae
    ],

    "MSE": [
        baseline_rmse**2,
        linear_rmse**2,
        best_poly_penalty["MSE"],
        rf_rmse**2,
        gb_rmse**2,
        mlp_rmse**2,
        mlp_lr_rmse**2,
        arima_rmse**2,
        sarimax_rmse**2
    ],

    "RMSE": [
        baseline_rmse,
        linear_rmse,
        best_poly_penalty["RMSE"],
        rf_rmse,
        gb_rmse,
        mlp_rmse,
        mlp_lr_rmse,
        arima_rmse,
        sarimax_rmse
    ],

    "R²": [
        baseline_r2,
        linear_r2,
        best_poly_penalty["R²"],
        rf_r2,
        gb_r2,
        mlp_r2,
        mlp_lr_r2,
        arima_r2,
        sarimax_r2
    ]
})

final_comparison_df = final_comparison_df.sort_values(
    "R²",
    ascending=False
)

print(final_comparison_df)

                                              Model       MAE        MSE  \
1                                 Linear Regression  0.673410   0.723264   
2  Polynomial Regression + Lasso Penalty (Degree 1)       NaN   0.727517   
3                                     Random Forest  0.830631   1.063995   
4                                 Gradient Boosting  0.962148   1.323468   
5                                               MLP  1.023208   1.750046   
0                                          Baseline  1.231944   2.217361   
6                           MLP + Linear Regression  1.217555   2.390229   
7                                             ARIMA  1.590245   3.568434   
8                                           SARIMAX  6.237075  48.614205   

       RMSE         R²  
1  0.850449   0.794614  
2  0.852946   0.793406  
3  1.031501   0.697856  
4  1.150421   0.624173  
5  1.322893   0.503037  
0  1.489081   0.370333  
6  1.546036   0.321243  
7  1.889030  -0.013334  
8  6.972389 -1

In [ ]:
#Final comparison table:

if "MAE" in poly_penalty_results_df.columns:
  final_comparison_df.loc[
      final_comparison_df["Model"].str.contains("Polynomial"),
      "MAE"
  ] = best_poly_penalty["MAE"]

#Sort by best R² value:
final_comparison_df = final_comparison_df.sort_values("R²", ascending=False)

#Display final table:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

final_comparison_df

,Model,MAE,MSE,RMSE,R²
1,Linear Regression,0.673410,0.723264,0.850449,0.794614
2,Polynomial Regression + Lasso Penalty (Degree 1),0.681950,0.727517,0.852946,0.793406
3,Random Forest,0.830631,1.063995,1.031501,0.697856
4,Gradient Boosting,0.962148,1.323468,1.150421,0.624173
5,MLP,1.023208,1.750046,1.322893,0.503037
0,Baseline,1.231944,2.217361,1.489081,0.370333
6,MLP + Linear Regression,1.217555,2.390229,1.546036,0.321243
7,ARIMA,1.590245,3.568434,1.889030,-0.013334
8,SARIMAX,6.237075,48.614205,6.972389,-12.805051


In [ ]:
#Print out final results of the Polynomial Regression model with Ridge + Lasso to see comparisons
poly_penalty_results_df.sort_values("RMSE")

,Penalty,Degree,MAE,MSE,RMSE,R²
1,Lasso,1,0.681950,0.727517,0.852946,0.793406
2,Ridge,2,0.776138,0.919240,0.958770,0.738962
0,Ridge,1,0.791423,0.990235,0.995106,0.718801
3,Lasso,2,0.966914,1.391779,1.179737,0.604774
4,Ridge,3,1.043508,1.711606,1.308284,0.513953
5,Lasso,3,1.118471,1.921883,1.386320,0.454240
7,Lasso,4,1.546112,3.531585,1.879251,-0.002870
6,Ridge,4,1.625412,4.197962,2.048893,-0.192102


In [ ]:
#Convolutional Neural Network:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#Start by reshaping all the data for the model. A CNN expects samples, timesteps, and channels.
X_train_cnn = np.array(X_train_scaled).reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_test_cnn = np.array(X_test_scaled).reshape(X_test_scaled.shape[0], X_test_scaled.shape[1], 1)

#Build the regression model for the CNN
cnn_model = Sequential([
    Conv1D(filters=32, kernel_size=3, activation="relu", input_shape=(X_train_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),

    Conv1D(filters=64, kernel_size=3, activation="relu"),
    Dropout(0.2),

    Conv1D(filters=128, kernel_size=3, activation="relu"),
    Dropout(0.2),

    Flatten(),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(64, activation="relu"),
    Dropout(0.2),

    Dense(32, activation="relu"),
    Dense(1)
])

cnn_model.compile(
    optimizer="adam",
    loss="mse"
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

history = cnn_model.fit(
    X_train_cnn,
    y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

cnn_preds = cnn_model.predict(X_test_cnn).flatten()

cnn_mae = mean_absolute_error(y_test, cnn_preds)
cnn_mse = mean_squared_error(y_test, cnn_preds)
cnn_rmse = np.sqrt(cnn_mse)
cnn_r2 = r2_score(y_test, cnn_preds)

print("CNN Results:")
print("MAE:", cnn_mae)
print("MSE", cnn_mse)
print("RMSE", cnn_rmse)
print("R²", cnn_r2)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 20ms/step - loss: 18.5278 - val_loss: 5.0180
Epoch 2/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 9.1258 - val_loss: 4.6979
Epoch 3/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 7.7946 - val_loss: 5.1637
Epoch 4/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 7.0412 - val_loss: 6.6850
Epoch 5/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 6.5344 - val_loss: 5.9410
Epoch 6/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 5.5329 - val_loss: 6.8634
Epoch 7/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 5.4566 - val_loss: 7.3359
Epoch 8/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 4.6853 - val_loss: 7.4003
Epoch 9/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.6323 - val_loss: 4.8105
Epoch 10/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 4.6356 - val_loss: 5.5339
Epoch 11/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 4.1256 - val_loss: 13.4218
Epoch 12/200
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss:

In [ ]:
# Zero-shot rule-based model
# This model is not trained on the dataset.
# It uses fixed assumptions to predict PM2.5 from environmental features.

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

zero_shot_df = test_df.copy()

# Start with persistence from the previous day
zero_shot_preds = zero_shot_df["pm25_lag_1"].copy()

# Add wildfire effect if available
if "fire_count" in zero_shot_df.columns:
    zero_shot_preds += 0.15 * zero_shot_df["fire_count"]

if "mean_frp" in zero_shot_df.columns:
    zero_shot_preds += 0.05 * zero_shot_df["mean_frp"]

if "max_frp" in zero_shot_df.columns:
    zero_shot_preds += 0.02 * zero_shot_df["max_frp"]

# Add wind effect if available
if "wind_speed_10m" in zero_shot_df.columns:
    zero_shot_preds += 0.10 * zero_shot_df["wind_speed_10m"]

# Add temperature effect if available
if "temp_2m_C" in zero_shot_df.columns:
    zero_shot_preds += 0.03 * zero_shot_df["temp_2m_C"]

# Reduce PM2.5 if NDVI is higher
if "NDVI" in zero_shot_df.columns:
    zero_shot_preds -= 2.0 * zero_shot_df["NDVI"]

if "mean_ndvi" in zero_shot_df.columns:
    zero_shot_preds -= 2.0 * zero_shot_df["mean_ndvi"]

# Prevent impossible negative PM2.5 predictions
zero_shot_preds = np.maximum(zero_shot_preds, 0)

# Evaluate
zero_shot_mae = mean_absolute_error(y_test, zero_shot_preds)
zero_shot_mse = mean_squared_error(y_test, zero_shot_preds)
zero_shot_rmse = np.sqrt(zero_shot_mse)
zero_shot_r2 = r2_score(y_test, zero_shot_preds)

print("Zero-Shot Rule-Based Model Results")
print("MAE:", zero_shot_mae)
print("MSE:", zero_shot_mse)
print("RMSE:", zero_shot_rmse)
print("R²:", zero_shot_r2)

Zero-Shot Rule-Based Model Results
MAE: 6.006944444444445
MSE: 39.60486111111111
RMSE: 6.29323931780058
R²: -10.2466537206901


In [ ]:
zero_shot_row = pd.DataFrame({
    "Model": ["Zero-Shot Rule-Based Model"],
    "MAE": [zero_shot_mae],
    "MSE": [zero_shot_mse],
    "RMSE": [zero_shot_rmse],
    "R²": [zero_shot_r2]
})

final_comparison_with_zero_shot = pd.concat(
    [final_comparison_df, zero_shot_row],
    ignore_index=True
)

final_comparison_with_zero_shot = final_comparison_with_zero_shot.sort_values("R²", ascending=False)

final_comparison_with_zero_shot

,Model,MAE,MSE,RMSE,R²
0,Linear Regression,0.673410,0.723264,0.850449,0.794614
1,Polynomial Regression + Lasso Penalty (Degree 1),0.681950,0.727517,0.852946,0.793406
2,Random Forest,0.830631,1.063995,1.031501,0.697856
3,Gradient Boosting,0.962148,1.323468,1.150421,0.624173
4,MLP,1.023208,1.750046,1.322893,0.503037
5,Baseline,1.231944,2.217361,1.489081,0.370333
6,MLP + Linear Regression,1.217555,2.390229,1.546036,0.321243
7,ARIMA,1.590245,3.568434,1.889030,-0.013334
9,Zero-Shot Rule-Based Model,6.006944,39.604861,6.293239,-10.246654
8,SARIMAX,6.237075,48.614205,6.972389,-12.805051


In [ ]:
# Elastic Net Regression Model
# Combines Ridge and Lasso penalties

from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

elastic_model = ElasticNet(
    alpha=0.001,
    l1_ratio=0.5,
    max_iter=10000,
    random_state=42
)

elastic_model.fit(X_train_scaled, y_train)

elastic_preds = elastic_model.predict(X_test_scaled)

elastic_mae = mean_absolute_error(y_test, elastic_preds)
elastic_mse = mean_squared_error(y_test, elastic_preds)
elastic_rmse = np.sqrt(elastic_mse)
elastic_r2 = r2_score(y_test, elastic_preds)

print("Elastic Net Results")
print("MAE:", elastic_mae)
print("MSE:", elastic_mse)
print("RMSE:", elastic_rmse)
print("R²:", elastic_r2)

Elastic Net Results
MAE: 0.6960774419731306
MSE: 0.7590942535287333
RMSE: 0.8712601526115683
R²: 0.7844388296968859


In [ ]:
elastic_row = pd.DataFrame({
    "Model": ["Elastic Net"],
    "MAE": [elastic_mae],
    "MSE": [elastic_mse],
    "RMSE": [elastic_rmse],
    "R²": [elastic_r2]
})

final_comparison_with_elastic = pd.concat(
    [final_comparison_df, elastic_row],
    ignore_index=True
)

final_comparison_with_elastic = final_comparison_with_elastic.sort_values("R²", ascending=False)

final_comparison_with_elastic

,Model,MAE,MSE,RMSE,R²
0,Linear Regression,0.673410,0.723264,0.850449,0.794614
1,Polynomial Regression + Lasso Penalty (Degree 1),0.681950,0.727517,0.852946,0.793406
9,Elastic Net,0.696077,0.759094,0.871260,0.784439
2,Random Forest,0.830631,1.063995,1.031501,0.697856
3,Gradient Boosting,0.962148,1.323468,1.150421,0.624173
4,MLP,1.023208,1.750046,1.322893,0.503037
5,Baseline,1.231944,2.217361,1.489081,0.370333
6,MLP + Linear Regression,1.217555,2.390229,1.546036,0.321243
7,ARIMA,1.590245,3.568434,1.889030,-0.013334
8,SARIMAX,6.237075,48.614205,6.972389,-12.805051


In [ ]:
#LLM Few-Shot Prompting Model:
#This gives the LLM a few examples, then asks it to predict PM2.5 for new rows.

from transformers import pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import re

In [ ]:
#Load Text Generation LLM
llm = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens = 20
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
#Pick a few examples for the prompt:

few_shot_examples = train_df.sample(
    n=5,
    random_state = 42
)

In [ ]:
#Helper function to extract a number from LLM output:

def extract_number(text):
  match = re.search(r"[-+]?\d*\.\d+|\d+", text)
  if match:
    return float(match.group())

  return np.nan

In [ ]:
# Build few-shot prompt

def build_few_shot_prompt(test_row):
    prompt = """
You are predicting daily mean PM2.5 concentration in micrograms per cubic meter.
Use the examples below to learn the pattern.
Return only one number.

Examples:
"""

    for _, row in few_shot_examples.iterrows():
        prompt += f"""
Input:
Previous day PM2.5: {row.get("pm25_lag_1", "unknown")}
3-day PM2.5 lag: {row.get("pm25_lag_3", "unknown")}
7-day PM2.5 lag: {row.get("pm25_lag_7", "unknown")}
NDVI: {row.get("mean_ndvi", row.get("NDVI", "unknown"))}
Fire count: {row.get("fire_count", "unknown")}
Mean FRP: {row.get("mean_frp", row.get("frp_mean", "unknown"))}
Max FRP: {row.get("max_frp", row.get("frp_max", "unknown"))}
Temperature: {row.get("temp_2m_C", "unknown")}
Wind speed: {row.get("wind_speed_10m", row.get("wind_speed", "unknown"))}

Output:
{row[target_col]}
"""

    prompt += f"""
Now predict this case:

Input:
Previous day PM2.5: {test_row.get("pm25_lag_1", "unknown")}
3-day PM2.5 lag: {test_row.get("pm25_lag_3", "unknown")}
7-day PM2.5 lag: {test_row.get("pm25_lag_7", "unknown")}
NDVI: {test_row.get("mean_ndvi", test_row.get("NDVI", "unknown"))}
Fire count: {test_row.get("fire_count", "unknown")}
Mean FRP: {test_row.get("mean_frp", test_row.get("frp_mean", "unknown"))}
Max FRP: {test_row.get("max_frp", test_row.get("frp_max", "unknown"))}
Temperature: {test_row.get("temp_2m_C", "unknown")}
Wind speed: {test_row.get("wind_speed_10m", test_row.get("wind_speed", "unknown"))}

Output:
"""

    return prompt

In [ ]:
# Run LLM few-shot predictions
# Start small first because LLM inference is slow.

llm_few_shot_preds = []

for i, row in test_df.iterrows():
    prompt = build_few_shot_prompt(row)

    output = llm(prompt)[0]["generated_text"]

    # Only parse the part after the final Output:
    final_answer = output.split("Output:")[-1]

    pred = extract_number(final_answer)

    # fallback if model fails to output a number
    if np.isnan(pred):
        pred = row["pm25_lag_1"]

    llm_few_shot_preds.append(pred)

llm_few_shot_preds = np.array(llm_few_shot_preds)

print("Finished LLM few-shot predictions")

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/t

Finished LLM few-shot predictions


In [ ]:
# Evaluate LLM few-shot model

llm_few_shot_mae = mean_absolute_error(y_test, llm_few_shot_preds)
llm_few_shot_mse = mean_squared_error(y_test, llm_few_shot_preds)
llm_few_shot_rmse = np.sqrt(llm_few_shot_mse)
llm_few_shot_r2 = r2_score(y_test, llm_few_shot_preds)

print("LLM Few-Shot Model Results")
print("MAE:", llm_few_shot_mae)
print("MSE:", llm_few_shot_mse)
print("RMSE:", llm_few_shot_rmse)
print("R²:", llm_few_shot_r2)

LLM Few-Shot Model Results
MAE: 2.7902777777777774
MSE: 11.683194444444446
RMSE: 3.418068817979598
R²: -2.3176948127535217


In [ ]:
# Add LLM few-shot model to comparison table

llm_few_shot_row = pd.DataFrame({
    "Model": ["LLM Few-Shot Prompting"],
    "MAE": [llm_few_shot_mae],
    "MSE": [llm_few_shot_mse],
    "RMSE": [llm_few_shot_rmse],
    "R²": [llm_few_shot_r2]
})

final_comparison_with_llm = pd.concat(
    [final_comparison_df, llm_few_shot_row],
    ignore_index=True
)

final_comparison_with_llm = final_comparison_with_llm.sort_values("R²", ascending=False)

final_comparison_with_llm

,Model,MAE,MSE,RMSE,R²
0,Linear Regression,0.673410,0.723264,0.850449,0.794614
1,Polynomial Regression + Lasso Penalty (Degree 1),0.681950,0.727517,0.852946,0.793406
2,Random Forest,0.830631,1.063995,1.031501,0.697856
3,Gradient Boosting,0.962148,1.323468,1.150421,0.624173
4,MLP,1.023208,1.750046,1.322893,0.503037
5,Baseline,1.231944,2.217361,1.489081,0.370333
6,MLP + Linear Regression,1.217555,2.390229,1.546036,0.321243
7,ARIMA,1.590245,3.568434,1.889030,-0.013334
9,LLM Few-Shot Prompting,2.790278,11.683194,3.418069,-2.317695
8,SARIMAX,6.237075,48.614205,6.972389,-12.805051
